In [29]:
###############################################################################
# WaterTAP Copyright (c) 2020-2026, The Regents of the University of California,
# through Lawrence Berkeley National Laboratory, Oak Ridge National Laboratory,
# National Laboratory of the Rockies, and National Energy Technology
# Laboratory (subject to receipt of any required approvals from the U.S. Dept.
# of Energy). All rights reserved.
#
# Please see the files COPYRIGHT.md and LICENSE.md for full copyright and license
# information, respectively. These files are also available online at the URL
# "https://github.com/watertap-org/watertap/"
###############################################################################

# Flexible RO Example 2

This tutorial expands on the functionality seen the previous file. The key differences are:
* Energy is a surrogate of recovery and feed flowrate
* Pretreatment is UF with it's own surrogate function
* [Insert Other Key things]

The general flow is very similar to the other tutorial- however, there are some significant differences in the underlying unit models

## Facility Description
The system being modeled is the ARC facility in the Water Replenishment District (WRD) in wherever. It has 4 RO trains and 4 UF pumps, though only 3 UF pumps can be on at one time.

TODO: [Insert schematic of WRD system]
# <img src="WRD_facility_diagram.png" width="1000" height="680">


In this tutorial, we will first create functions for each of the major steps in the model creation, then compare the results for flexible operations under three different rate structures.

## Step 1: Import libraries and load price signals and PV power data

In [30]:
import os
import importlib.metadata

import pyomo.environ as pyo
import pytest
import pandas as pd

from idaes.apps.grid_integration import PriceTakerModel

from watertap.flowsheets.flex_desal import wrd_ro_flowsheet as fs
from watertap.flowsheets.flex_desal import utils
from watertap.flowsheets.flex_desal.params import FlexDesalParams
from watertap.core.solvers import get_solver
from idaes.core.util.model_statistics import degrees_of_freedom
solver = get_solver()


def load_price_data(csv_path):
    df = pd.read_csv(csv_path)
    df["Energy Rate"] = (
        df["electric_energy_on_peak"]
        + df["electric_energy_mid_peak"]
        + df["electric_energy_off_peak"]
        + df["electric_energy_super_off_peak"]
    )
    df["Fixed Demand Rate"] = df["electric_demand_fixed"]
    df["Var Demand Rate"] = df["electric_demand_peak"]
    df["Customer Cost"] = df["electric_customer_fixed_charge"]
    df["Demand_Response_Price"] = df["electric_demand_response_price"]
    df["Emissions Intensity"] = 0

    peak = df["Var Demand Rate"].to_numpy() != 0
    pv_kw = df["solar_output_kW"]
    pv_cap = max(pv_kw)
    pv_cf = pv_kw / pv_cap
    return df, peak, pv_kw, pv_cap, pv_cf


def set_price_data(csv_path):
    global price_data, peak_hours, pv_kW, pv_capacity, pv_capacity_factors
    (
        price_data,
        peak_hours,
        pv_kW,
        pv_capacity,
        pv_capacity_factors,
    ) = load_price_data(csv_path)


set_price_data("price_signals/wrd_pricesignal_summer_week.csv")

## Step 2: Create an instance of the PriceTakerModel, set parameters for the components, and build the multiperiod model
Major additions are different surrogate type, flexible flowrate, replacement costs, flow baswed costs for the unit models

In [31]:
def build_model():
    # Find start and end datetimes and time step  from the price data
    price_datetimes = pd.to_datetime(price_data["DateTime"])
    data_start = price_datetimes.iloc[0]
    data_next_time = price_datetimes.iloc[1]
    timestep_hours = (data_next_time - data_start).total_seconds() / 3600
    start_date = data_start.strftime("%Y-%m-%d %H:%M:%S")
    end_date = price_datetimes.iloc[-1].strftime("%Y-%m-%d %H:%M:%S")


    m = PriceTakerModel()
    m.params = FlexDesalParams(
        start_date=start_date,
        end_date=end_date,
        annual_production_AF=12000,
        timestep_hours=timestep_hours,
        include_onsite_solar=True,
        onsite_capacity=pv_capacity,
        CAPEX_yr=6498300,  # For WRD, this assumes a 30 yr lifetime
        include_demand_response=True, # For now, this value is 0
    )

    m.params.intake.update(
                {
                    "energy_intensity": 0,
                    "nominal_flowrate": 2500, # m3/hr # Note this is treated as an upper bound! (Should it be renamed??)
                    "feed_cost": 0.16, # $/m3
                    "chemical_cost": 0.0332, # $/m3 # pretreatment chemical costs
                }
            )  

    m.params.wrd_uf.update(
        {
            "minimum_downtime": 2, # hours
            "startup_delay": 2,
            "minimum_flowrate": 344,  # m3/hr
            "nominal_flowrate": 900,
            "maximum_flowrate": 989,
            "surrogate_type": "quadratic_energy_intensity", # kWh/m3 = a + b*flowrate + c*flowrate^2
            "surrogate_a": 2.71e-1,
            "surrogate_b": -3.32e-4,
            "surrogate_c": 2.39e-7,
            "nominal_recovery": 0.96, # fixed UF recovery
            "num_uf_pumps": 3, # plant has 4 pumps, but only a maximum of 3 on at one time
        }
    )

    m.params.wrd_ro.update(
        {
            "startup_delay": 2,  # hours
            "minimum_downtime": 2,  
            "minimum_flowrate": 520,  # m3/hr
            "nominal_flowrate": 602,
            "maximum_flowrate": 635,
            "allow_variable_recovery": True,
            "surrogate_type": "PySMO_polyfit", # Key difference is the surrogate is from PySMO
            "surrogate_file": "ro_SEC_poly_fit_order_2.json", # oder_1 can also be used and should improve the solve time
            "minimum_recovery": 0.88,
            "nominal_recovery": 0.925,
            "maximum_recovery": 0.925,
            "num_ro_skids": 4,
            "replacement_types": ["membranes", "motors"],
            "replacement_costs": [500 * 4 * (72 + 30 + 15), 125000],  # $ per replacement
            "replacement_lifetimes": [5, 20],  # years
            "replacement_max_flex_penalty": [0.1, 0.1],  # Reduction in lifetime if shutdowns occurs every day
        }
    ) # TODO: have some documentation on the replacement cost calculation. Then reference it here.

    # Postreatement is a UV system
    m.params.posttreatment.update(
        {
            "energy_intensity": 0.101, # kWh/m3
            "chemical_cost": 0.0310, # $/m3
        }
    ) 

    m.params.brinedischarge.update({"brine_cost": 0.43, "energy_intensity": 0})

    m.append_lmp_data(lmp_data=price_data["Energy Rate"])

    # The flowsheet build is in a different file
    m.build_multiperiod_model(
        flowsheet_func=fs.build_desal_flowsheet,
        flowsheet_options={"params": m.params},
    )

    return m

## Step 3: Add constraints from the flowsheet (fs) and fix operation variables that do not vary with time 

In [32]:
def add_constraints(m):
    m.update_operation_params(
        {
            "fixed_demand_rate": price_data["Fixed Demand Rate"],
            "variable_demand_rate": price_data["Var Demand Rate"],
            "emissions_intensity": price_data["Emissions Intensity"],
            "customer_cost": price_data["Customer Cost"],
            "demand_response_price": price_data["Demand_Response_Price"],
        }
    )

    m.update_operation_params(
        {"power_generation.capacity_factor": pv_capacity_factors}
    )

    # Add demand cost and fixed cost calculation constraints
    fs.add_demand_and_fixed_costs(m)

    # Add the startup delay constraints
    fs.add_delayed_startup_constraints(m)

    # Ensure consistent ending and starting states of the plant
    fs.begin_and_end_constraint(m)


## Step 4: Construct useful expressions or model-level constraints

In [33]:
def add_expressions(m):    
    # Add the demand response revenue calculations
    fs.add_useful_expressions(m)

    # Add costs for brine, feed, and chemical flows
    fs.add_flow_costs(m)

    m.total_water_production = pyo.Expression(
        expr=m.params.timestep_hours
        * sum(m.period[:, :].posttreatment.product_flowrate)
        )

    m.total_energy_cost = pyo.Expression(expr=sum(m.period[:, :].energy_cost))

    # Demand costs are automatically normalized by number of months. So for a sample week, it multiplies by 7 / 31.
    m.total_demand_cost = pyo.Expression(expr=m.fixed_demand_cost + m.variable_demand_cost)

    m.total_customer_cost = pyo.Expression(expr=sum(m.period[:, :].customer_cost)*m.params.num_months)

    m.total_op_cost = pyo.Expression(
        expr=m.total_energy_cost
        + m.total_demand_cost
        + m.total_customer_cost
        - m.total_demand_response_revenue
        + m.total_feed_cost
        + m.total_brine_cost
        + m.total_chemical_cost
        )

    # add CAPEX as a fixed cost to calculate LCOW
    m.fixed_cost = pyo.Expression(expr=m.params.CAPEX_yr * m.params.num_months / 12)
    m.total_cost = pyo.Expression(expr=m.total_op_cost + m.fixed_cost)

    m.LCOW = pyo.Expression(expr=m.total_cost / m.total_water_production)  # $/m3

    # fix the UF recovery
    utils.wrd_fix_uf_recovery(m, uf_recovery=m.params.wrd_uf.nominal_recovery)

    fs.constrain_water_production(m)


## Step 5: Define a penalty term for changing the flowrate between hours
This makes a distinction between the on-on-off-off and on-off-on-off

It also allows the solver to find cleaner solutions, instead of showing +/- 5% flowrate fluctations betweeen hours.

*Not immediately obvious why you would want a penalty term for this, so more explanation can be included if needed*

In [34]:
def add_flow_changes_penalty(m):
    # Adds m.flow_changes_penalty, which is proportional to the number of times an RO train or UF pump changes flowrate between hours 
    # Applied to both the RO and UF systems
    fs.add_flow_changes_penalty_binary(m)

## Step 6: Add objective function to minimize operating costs

In [35]:
def add_objective(m):
    m.obj = pyo.Objective(
                expr=1e-4
                * (
                    m.total_energy_cost
                    + m.total_demand_cost
                    + m.total_customer_cost
                    - m.total_demand_response_revenue
                    + m.total_feed_cost
                    + m.total_brine_cost
                    + m.total_chemical_cost
                    + m.flow_changes_penalty
                ),
                sense=pyo.minimize,
            )

## Step 7: Use solver

In [36]:
def solve_PT_model(m):
    # These could also be inputs to the function instead
    solver_name = "gurobi"
    # solver_name = "baron"
    # solver_name = "scip"
    # solver_name = "cplex"
    mip_gap = 0.025 # 2.5% MIP gap 

    if solver_name == "gurobi":
        # solver = utils.get_gurobi_solver_model(m, mip_gap=0.005)
        solver = pyo.SolverFactory("gurobi_direct_minlp")
        solver.options["MIPGap"] = mip_gap
        results = solver.solve(m, tee=True)
    # HAVEN"T TESTED WITH THESE SOLVERS YET
    # elif solver_name == "scip":
    #     solver = pyo.SolverFactory("scip", validate=False)
    #     results = solver.solve(m, tee=True)

    # elif solver_name in ["baron", "cplex"]:
    #     solver = pyo.SolverFactory("gams")
    #     results = solver.solve(
    #         m, tee=True, solver=solver_name, add_options=[f"options optcr={mip_gap};"]
    #     )

    pyo.assert_optimal_termination(results)

## Step 8: Perform the post solve calculations
Note- The replacement costs could be added to the operational costs earlier (and therefore the objective function). However, due to the formulation, it increases the non-lineararity and computational difficulty of the problem, even though it is a tiny << 1% of the total costs, so it doesn't drive optimization in this case study. Therefore, it is better to calculate it after the solve. It is a function of the flexibility or how many start-up and shutdowns occur during the week.

In [37]:
def post_solve_calculations(m):
    # Calculate the replacement costs
    fs.calculate_replacement_costs(m)

    # Calculate the flexibility metrics
    # TODO: Double check these baseline numbers
    fs.calculate_flexibility_metrics(
            m,
            baseline_power=1080,
            baseline_electricity_cost=50843,  # $/kWh
            baseline_replacement_cost=992,
        )

    design_var_values = m.get_design_var_values()
    filtered_design_var_values = {
        k: v
        for k, v in design_var_values.items()
        if "flow_change" not in k and "flow_changed" not in k and "reduction" not in k
    }

## Step 8: Save outputs and plot operations

In [38]:
def outputs(m, output_name):    
    m.get_operation_var_values().to_csv(f"{output_name}.csv")
    print(f"Saved operation variable results to: {output_name}.csv")

    utils.plot_function(
        m,
        n_time_points=len(price_data),
        output_stem=f"{output_name}",
        peak_hours=peak_hours,
    )

## Run the model

In [39]:
def run_model(result_name="wrd_result"):
    m = build_model()
    add_constraints(m)
    add_expressions(m)
    add_flow_changes_penalty(m)
    add_objective(m)

    try:
        solve_PT_model(m)
    except Exception as e:
        error_message = str(e)
        if "gurobi_direct_minlp" in error_message and "not available" in error_message:
            print("\nDetected unavailable solver: gurobi_direct_minlp\n")
            # Run specific fallback code only for this solver-not-found error.
            # Replace the line below with your custom fallback logic.
            print("Solving with ipopt instead for testing purposes, but the solution won't make any sense.")
            solver = get_solver()
            results = solver.solve(m, tee=True)
            pyo.assert_optimal_termination(results)
        else:
            raise

    post_solve_calculations(m)
    outputs(m, result_name) # This doesn't output the plot to the notebook, but saves it as a file instead. Maybe that's fine?

## Example Analysis: Comparison of different rate structures

This tool can be used to evaluate how different rate structures influnce electricity costs and flexible operations. 

In [40]:
# Run the model with the exisitng rate structure
run_model()

(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block
pretreatment with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
2026-06-18 18:14:52 [INFO] idaes.core.surrogate.pysmo_surrogate: Decode surrogate. type=poly

===========================Polynomial Regression===============================================

The number of cross-validation cases (3) is used.
The default training/cross-validation split of 0.75 is used.
No iterations will be run.
Default parameter estimation method is used.
Parameter estimation method:  pyomo 

2026-06-18 18:14:52 [INFO] idaes.core.surrogate.pysmo_surrogate: Decode surrogate. type=poly

===========================Polynomial Regression===============================================

The number of cross-validation cases (3) is used.
The default training/cross-validation split of 0.75 is

C:\Users\rchurchi\watertap\watertap\flowsheets\flex_desal\utils.py:413: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
# Changing the rate structure to Real Time Pricing (RTP)
set_price_data("price_signals/wrd_pricesignal_summer_week_hot_RTP.csv")

run_model(result_name="wrd_result_RTP")

(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block
pretreatment with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
2026-06-18 18:16:15 [INFO] idaes.core.surrogate.pysmo_surrogate: Decode surrogate. type=poly

===========================Polynomial Regression===============================================

The number of cross-validation cases (3) is used.
The default training/cross-validation split of 0.75 is used.
No iterations will be run.
Default parameter estimation method is used.
Parameter estimation method:  pyomo 

2026-06-18 18:16:15 [INFO] idaes.core.surrogate.pysmo_surrogate: Decode surrogate. type=poly

===========================Polynomial Regression===============================================

The number of cross-validation cases (3) is used.
The default training/cross-validation split of 0.75 is

C:\Users\rchurchi\watertap\watertap\flowsheets\flex_desal\utils.py:413: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [42]:
# Maintaining same rate structure to include a demand response event
set_price_data("price_signals/wrd_pricesignal_summer_week_DR.csv")

run_model(result_name="wrd_result_DR")

(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block
pretreatment with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().
2026-06-18 18:17:33 [INFO] idaes.core.surrogate.pysmo_surrogate: Decode surrogate. type=poly

===========================Polynomial Regression===============================================

The number of cross-validation cases (3) is used.
The default training/cross-validation split of 0.75 is used.
No iterations will be run.
Default parameter estimation method is used.
Parameter estimation method:  pyomo 

2026-06-18 18:17:33 [INFO] idaes.core.surrogate.pysmo_surrogate: Decode surrogate. type=poly

===========================Polynomial Regression===============================================

The number of cross-validation cases (3) is used.
The default training/cross-validation split of 0.75 is

C:\Users\rchurchi\watertap\watertap\flowsheets\flex_desal\utils.py:413: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Compare the three plots 

### Existing Rate Structure
<img src="wrd_result.png" width="1000" height="680">

### Demand Response Event
<img src="wrd_result_DR.png" width="1000" height="680">

### Real-Time Pricing
<img src="wrd_result_RTP.png" width="1000" height="680">

Takeaway: Optimal operations depend on the price signals. An estimate of which rate structure would be best for the facility depends on the ability to do flexible operations

## Other helpful functions

There are a number of other constraints that could be considered. Though not included in this example, the functions can be found in flowsheet and are described below.

In [43]:
# Fixed Recovery

# Working Hours

# Rainy Days

print("Last Cell")

Last Cell
